# Project - Ethics of AI
## | Make Ethical a Classic AI Workflow |

---

## Downloading

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np
import random
from scipy.stats import chi2_contingency


---

## Data Ingestion

### Import

In [ ]:
df_init = pd.read_csv("../data/Loan.csv")
random.seed(7)

### Cleaning

In [ ]:
df_init_proper = df_init[
    [
        "Age", 
        "MaritalStatus", 
        "EmploymentStatus", 
        "EducationLevel", 
        "Experience", 
        "MonthlyLoanPayment", 
        "MonthlyIncome", 
        "UtilityBillsPaymentHistory",
        "LoanApproved"
    ]
]

### Exploration

In [ ]:
display(df_init_proper)
print(df_init_proper.dtypes)

In [ ]:
target = "LoanApproved"

# Detect categorical and numerical columns
categorical_cols = df_init_proper.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = df_init_proper.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove label col from those lists if present
categorical_cols = [c for c in categorical_cols if c != target]
numerical_cols = [c for c in numerical_cols if c != target]

# --- PLOTS ---

for col in categorical_cols:
    plt.figure(figsize=(6,4))
    sns.countplot(data=df_init_proper, x=col, hue=target)
    plt.title(f'Distribution of {col} by {target}')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

for col in numerical_cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(data=df_init_proper, x=target, y=col)
    plt.title(f'Distribution of {col} by {target}')
    plt.tight_layout()
    plt.show()

if pd.api.types.is_numeric_dtype(df_init_proper[target]):
    corr = df_init_proper[numerical_cols + [target]].corr()[target].sort_values(ascending=False)
    plt.figure(figsize=(5,5))
    sns.barplot(x=corr.values, y=corr.index)
    plt.title(f'Correlation of features with {target}')
    plt.show()

---

## Biases 

In [ ]:
def detect_bias(df, feature, target='LoanApproved'):
    stats = df.groupby(feature)[target].agg(['sum', 'count', 'mean'])
    stats.columns = ['Approved', 'Total', 'Approval_Rate']
    stats['Global_Rate'] = df[target].mean()
    stats['Gap_%'] = (stats['Approval_Rate'] - stats['Global_Rate']) * 100
    
    contingency = pd.crosstab(df[feature], df[target])
    chi2, p_value, _, _ = chi2_contingency(contingency)
    stats['P_Value'] = p_value
    
    return stats.sort_values('Approval_Rate', ascending=False)

In [ ]:
def disparate_impact(df, feature, target='LoanApproved'):
    rates = df.groupby(feature)[target].mean()
    max_rate = rates.max()
    
    results = []
    for group in rates.index:
        ratio = rates[group] / max_rate
        results.append({
            'Group': group,
            'Rate': rates[group],
            'Ratio': ratio,
            'Pass_80%': ratio >= 0.8
        })
    
    return pd.DataFrame(results)

In [ ]:
df = pd.read_csv("../data/Loan.csv")

age_bins = [18, 25, 35, 45, 55, 65, 100]
age_labels = ['18-25', '26-35', '36-45', '46-55', '56-65', '65+']
df['AgeGroup'] = pd.cut(df['Age'], bins=age_bins, labels=age_labels)

print("Bias detection:")
bias_stats = detect_bias(df, 'AgeGroup')
display(bias_stats)

print("\nDisparate impact:")
di_results = disparate_impact(df, 'AgeGroup')
display(di_results)

In [ ]:
df = pd.read_csv("../data/Loan.csv")

age_bins = [18, 25, 35, 45, 55, 65, 100]
age_labels = ['18-25', '26-35', '36-45', '46-55', '56-65', '65+']
df['AgeGroup'] = pd.cut(df['Age'], bins=age_bins, labels=age_labels)

# 2. Analyse statistique du biais (tableaux)
print("Bias detection:")
bias_stats = detect_bias(df, 'AgeGroup')
display(bias_stats)

print("\nDisparate impact:")
di_results = disparate_impact(df, 'AgeGroup')
display(di_results)

# 3. VISUALISATION GRAPHIQUE DU BIAIS (À AJOUTER ICI)
visualize_bias(df, 'AgeGroup')  # ← Sur les données ORIGINALE

In [ ]:
def reweighting(df, target='LoanApproved'):
    df_temp = df.copy()
    df_temp = df_temp.dropna(subset=['Age', target])
    
    age_bins = [18, 25, 35, 45, 55, 65, 100]
    age_labels = ['18-25', '26-35', '36-45', '46-55', '56-65', '65+']
    df_temp['AgeGroup'] = pd.cut(df_temp['Age'], bins=age_bins, labels=age_labels)
    df_temp = df_temp.dropna(subset=['AgeGroup'])
    
    total = len(df_temp)
    n_groups = df_temp['AgeGroup'].nunique()
    n_labels = df_temp[target].nunique()
    
    weights = []
    for _, row in df_temp.iterrows():
        group = row['AgeGroup']
        label = row[target]
        
        n = len(df_temp[(df_temp['AgeGroup'] == group) & (df_temp[target] == label)])
        expected = total / (n_groups * n_labels)
        weight = expected / n if n > 0 else 1.0
        weights.append(weight)
    
    df_temp['sample_weight'] = weights
    
    print("Reweighting applied")
    print(f"Samples: {len(df_temp)}")
    print(f"Weight range: [{df_temp['sample_weight'].min():.3f}, {df_temp['sample_weight'].max():.3f}]")
    
    return df_temp

df_reweighted = reweighting(df)
display(df_reweighted.groupby('AgeGroup')['sample_weight'].agg(['mean', 'count']))


In [ ]:
def resampling(df, target='LoanApproved', strategy='oversample'):
    df_temp = df.copy()
    df_temp = df_temp.dropna(subset=['Age', target])
    
    age_bins = [18, 25, 35, 45, 55, 65, 100]
    age_labels = ['18-25', '26-35', '36-45', '46-55', '56-65', '65+']
    df_temp['AgeGroup'] = pd.cut(df_temp['Age'], bins=age_bins, labels=age_labels)
    df_temp = df_temp.dropna(subset=['AgeGroup'])
    
    group_counts = df_temp.groupby(['AgeGroup', target]).size()
    
    if strategy == 'oversample':
        max_count = group_counts.max()
        
        resampled_dfs = []
        for (group, label), count in group_counts.items():
            subset = df_temp[(df_temp['AgeGroup'] == group) & (df_temp[target] == label)]
            n_samples = max_count - count
            if n_samples > 0:
                oversampled = subset.sample(n=n_samples, replace=True, random_state=42)
                resampled_dfs.append(subset)
                resampled_dfs.append(oversampled)
            else:
                resampled_dfs.append(subset)
        
        df_resampled = pd.concat(resampled_dfs, ignore_index=True)
    
    elif strategy == 'undersample':
        min_count = group_counts.min()
        
        resampled_dfs = []
        for (group, label), count in group_counts.items():
            subset = df_temp[(df_temp['AgeGroup'] == group) & (df_temp[target] == label)]
            undersampled = subset.sample(n=min_count, replace=False, random_state=42)
            resampled_dfs.append(undersampled)
        
        df_resampled = pd.concat(resampled_dfs, ignore_index=True)
    
    print(f"Resampling applied: {strategy}")
    print(f"Original samples: {len(df_temp)}")
    print(f"Resampled samples: {len(df_resampled)}")
    
    return df_resampled

df_oversampled = resampling(df, strategy='oversample')
display(df_oversampled.groupby('AgeGroup')[target].value_counts().unstack())

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
from sklearn.preprocessing import StandardScaler

def fairness_constraints_normalized(df, target='LoanApproved'):
    df_temp = df.copy()
    df_temp = df_temp.dropna(subset=['Age', target])
    
    age_bins = [18, 25, 35, 45, 55, 65, 100]
    age_labels = ['18-25', '26-35', '36-45', '46-55', '56-65', '65+']
    df_temp['AgeGroup'] = pd.cut(df_temp['Age'], bins=age_bins, labels=age_labels)
    df_temp = df_temp.dropna(subset=['AgeGroup'])
    
    numerical_features = ['AnnualIncome', 'CreditScore', 'DebtToIncomeRatio']
    available_features = [f for f in numerical_features if f in df_temp.columns]
    
    X = df_temp[available_features].fillna(0)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    y = df_temp[target]
    groups = df_temp['AgeGroup']
    
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_scaled, y)
    predictions = model.predict(X_scaled)
    
    print("Fairness constraints with normalization")
    print(f"Model accuracy: {accuracy_score(y, predictions):.3f}")
    
    print("\nPrediction rates by age group:")
    results = []
    for group in sorted(groups.unique()):
        mask = groups == group
        pred_rate = predictions[mask].mean()
        actual_rate = y[mask].mean()
        results.append({
            'Group': group,
            'Predicted_Rate': pred_rate,
            'Actual_Rate': actual_rate,
            'Gap': abs(pred_rate - actual_rate)
        })
    
    results_df = pd.DataFrame(results)
    display(results_df)
    
    return model, scaler

model_fair, scaler = fairness_constraints_normalized(df)

In [ ]:
def compare_mitigation_results(df, target='LoanApproved'):
   
    print("Bias technique comparison")
   
    
    age_bins = [18, 25, 35, 45, 55, 65, 100]
    age_labels = ['18-25', '26-35', '36-45', '46-55', '56-65', '65+']
    df['AgeGroup'] = pd.cut(df['Age'], bins=age_bins, labels=age_labels)
    
    print("\n1. Original bias:")
    print("-"*70)
    original = detect_bias(df, 'AgeGroup', target)
    display(original[['Approval_Rate', 'Gap_%']])
    
    di_original = disparate_impact(df, 'AgeGroup', target)
    fails_original = len(di_original[~di_original['Pass_80%']])
    print(f"\nGroups failing 80% rule: {fails_original}/6")
    
    print("\n2. after  REWEIGHTING:")
    print("-"*70)
    df_rw = reweighting(df, target)
    print("Sample weights computed to balance age groups")
    
    print("\n3. after RESAMPLING:")
    print("-"*70)
    df_rs = resampling(df, target, strategy='oversample')
    resampled_stats = detect_bias(df_rs, 'AgeGroup', target)
    display(resampled_stats[['Approval_Rate', 'Gap_%']])
    
    di_resampled = disparate_impact(df_rs, 'AgeGroup', target)
    fails_resampled = len(di_resampled[~di_resampled['Pass_80%']])
    print(f"\nGroups failing 80% rule: {fails_resampled}/6")
    
    print("\n4. FAIRNESS CONSTRAINTS:")
   
    print("Model trained without Age feature")
    print("Uses only: Income, CreditScore, DebtToIncomeRatio")
    
    print("Summary")

    print(f"Original failing groups: {fails_original}/6")
    print(f"After resampling: {fails_resampled}/6")
    print("Fairness constraints: Age feature removed from model")

compare_mitigation_results(df)

In [ ]:
print("Preparing data with bias mitigation: Resampling")
df_mitigated = resampling(df, target='LoanApproved', strategy='oversample')
print(f"Resampled data ready: {len(df_mitigated)} samples")
print("Age feature kept in dataset")

---

## Fine-Tuning

**Importing libraries**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, roc_curve
import os

import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer, set_seed
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate

import matplotlib.pyplot as plt

**Configurate the fine-tuning**
* MAX_LEN : Random seed for reproducibility
* EPOCHS : Steps for training operation
* BATCH : Number of samples processed together in one training step
* MODEL : prajjwal1/bert-tiny

*Which method did we choose ?*
* We plan to implement the full fine-tune method because of its good global performance and the small size of our dataset

*Why we decided to use this model?*
* Not heavy
* Possibility of a good trade-off between fine-tuning performance and memory usage

In [ ]:
SEED = 42
MODEL_NAME = "prajjwal1/bert-tiny"
MAX_LEN = 96
EPOCHS = 2
BATCH = 16

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

**Importing Data**

* Dropping useless features to enhance performance  
* Splitting features / target  
* Converting tabular data into textual to be handle by the LLM    

In [ ]:
df = df_mitigated.copy()

target = "LoanApproved"

feature_cols = [c for c in df.columns if c != target and c not in ['AgeGroup', 'sample_weight']]

def row_to_text(row, cols):
    parts = []
    for c in cols:
        v = row[c]
        if pd.api.types.is_numeric_dtype(df[c]):
            if isinstance(v, float):
                v = round(v, 3)
            else:
                v = int(v)
        parts.append(f"{c}: {v}")
    return " | ".join(parts)

print(f"Training with resampled data: {len(df)} samples")
print(f"Features: {len(feature_cols)}")
print(f"Age feature: KEPT (bias mitigated via resampling)")

**Splitting trainings and testing sets**
* Casting the target
* Setting up test_size to 30%

In [ ]:
texts = [row_to_text(r, feature_cols) for r in df[feature_cols].to_dict(orient="records")]
y = df[target].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(texts, y, test_size=0.30, random_state=SEED)

print(f"Training set lenght: {len(X_train)}")
print(f"Testing set lenght: {len(X_test)}")

print("Exemple of training data:\n")
print(X_train[0])

**Tokenizing and Building Hugging Face Datasets**

* Load the tokenizer linked to the chosen pretrained model   
* Provide a more-readable format for tensors across the encode batch function   
* Create Hugging Face Dataset    
* Padding each batch to the longest sequence in it with the collator (avoir to fix max_lenght and then enhance performance)

In [ ]:
path_to_model = "loan_model/"

if os.path.exists(path_to_model) :
    print("A tockenizer already exist")
    tok = AutoTokenizer.from_pretrained(path_to_model)
else :
    print("No tockenizer identify")
    tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

In [ ]:
def encode_batch(batch):
    """
    Function that converts text and labels into model-readable tensors
    """
    enc = tok(batch["text"], truncation=True, padding=False, max_length=MAX_LEN)
    enc['labels'] = batch['labels']
    return enc

## Full:

In [ ]:
# Build the Hugging Face Datasets
ds_train = Dataset.from_dict({"text": X_train, "labels": y_train}).map(encode_batch, batched=True)
ds_test  = Dataset.from_dict({"text": X_test,  "labels": y_test}).map(encode_batch, batched=True)

# Data collator handles dynamic padding per batch
collator = DataCollatorWithPadding(tokenizer=tok)

**Fine-tuning step**
* Setting up the model, training arguments and the trainer
* Launching the training

**NB : this step will cost you 3 minutes of your life :)**

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

if os.path.exists(path_to_model) :
    print("A model already exist")
    model = AutoModelForSequenceClassification.from_pretrained(path_to_model)
else :
    print("No model already train.")
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2) 

In [ ]:
# Only to train the model again

args = TrainingArguments(
    output_dir="outputs",
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH,
    num_train_epochs=EPOCHS,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    data_collator=collator,
    tokenizer=tok,
)

trainer.train()

**Applying the fine-tuned model**
* Applying AUC Score and F1-Score to evaluate the model
* Plotting the AUC Score for more visibility

In [ ]:
def plot_roc_auc(y_true, y_prob, title_prefix="Test"):
    """
    y_true: 1D array-like of true labels (0/1)
    y_prob: 1D array-like of predicted probabilities for class 1
    """
    auc = roc_auc_score(y_true, y_prob)
    fpr, tpr, _ = roc_curve(y_true, y_prob)

    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
    plt.plot([0,1], [0,1], linestyle="--", label="Chance")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{title_prefix} ROC Curve")
    plt.legend(loc="lower right")
    plt.show()
    return auc

In [ ]:
preds = trainer.predict(ds_test)
logits = preds.predictions
probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
print("ROC-AUC:", roc_auc_score(y_test, probs))

In [ ]:
auc = plot_roc_auc(y_test, probs, title_prefix="Loan")
print("ROC-AUC:", auc)

**Saving the model**

In [ ]:
# Save the model weights and configuration
model.save_pretrained("loan_model")
# Save the tokenizer (you need it to encode text again later)
tok.save_pretrained("loan_model")
print("Model and tokenizer saved in folder 'loan_model/'")

**Conclusion**

* The model can easily distinguish between people who can and cannot get a loan due to predictive power from the features
* It also successfully adapted from the original tabular format. Then we can say full fine-tuning worked well
* AUC score reveals that the ranking of positives vs negatives is almost perfect

This model could be optimized by :
* Adding some features we removed to better understand causality of a loan not given
* Adding more epochs
* Trying to get a bigger dataset and apply other methods (LoRa, Distillation)

## LoRA:

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from transformers import TrainingArguments, Trainer

base_model_path = "loan_model"
model = AutoModelForSequenceClassification.from_pretrained(base_model_path)
tok = AutoTokenizer.from_pretrained(base_model_path)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=32, 
    lora_dropout=0.1,
    bias="none", 
    target_modules=["query", "value"],  # Layers to inject LoRA into (for BERT)
)

lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()

lora_args = TrainingArguments(
    output_dir="outputs_lora",
    learning_rate=1e-4,
    per_device_train_batch_size=BATCH,
    num_train_epochs=EPOCHS,
    save_strategy="no",
    report_to="none",
)

lora_trainer = Trainer(
    model=lora_model,
    args=lora_args,
    train_dataset=ds_train,
    eval_dataset=ds_test,
    data_collator=collator,
    tokenizer=tok,
)

lora_trainer.train()


lora_preds = lora_trainer.predict(ds_test)
lora_logits = lora_preds.predictions
lora_probs = torch.softmax(torch.tensor(lora_logits), dim=1)[:, 1].numpy()

auc_lora = plot_roc_auc(y_test, lora_probs, title_prefix="Loan - LoRA")
print(f"✅ LoRA Fine-tuning completed — ROC-AUC: {auc_lora:.4f}")

lora_save_dir = "loan_model_lora"
lora_model.save_pretrained(lora_save_dir)
tok.save_pretrained(lora_save_dir)
print(f"LoRA adapters and tokenizer saved in folder '{lora_save_dir}/'")

## Distillation:

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
import torch.nn.functional as F
import numpy as np
import evaluate

# Load teacher (frozen)
teacher = AutoModelForSequenceClassification.from_pretrained("loan_model").eval()
for p in teacher.parameters():
    p.requires_grad = False

# Load smaller student
student = AutoModelForSequenceClassification.from_pretrained("prajjwal1/bert-tiny", num_labels=2)


temperature = 2.0
alpha = 0.7
metric = evaluate.load("accuracy")

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": metric.compute(predictions=preds, references=p.label_ids)["accuracy"]}

class DistillationTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"]
        outputs_s = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits_s = outputs_s.logits

        with torch.no_grad():
            outputs_t = teacher(**{k: v for k, v in inputs.items() if k != "labels"})
            logits_t = outputs_t.logits

        # Hard and soft losses
        loss_ce = F.cross_entropy(logits_s, labels)
        loss_kl = F.kl_div(
            F.log_softmax(logits_s / temperature, dim=-1),
            F.softmax(logits_t / temperature, dim=-1),
            reduction="batchmean"
        ) * (temperature ** 2)

        loss = alpha * loss_kl + (1 - alpha) * loss_ce
        return (loss, outputs_s) if return_outputs else loss


args = TrainingArguments(
    output_dir="loan_model_distilled",
    learning_rate=5e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",      
    load_best_model_at_end=True,  
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = DistillationTrainer(
    model=student,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_test,
    tokenizer=tok,
    compute_metrics=compute_metrics
)

trainer.train()

trainer.save_model("loan_model_distilled")
tok.save_pretrained("loan_model_distilled")
print("✅ Distilled model saved to 'loan_model_distilled/'")

lora_preds = trainer.predict(ds_test)
lora_logits = lora_preds.predictions
lora_probs = torch.softmax(torch.tensor(lora_logits), dim=1)[:, 1].numpy()

auc_distilled = plot_roc_auc(y_test, lora_probs, title_prefix="Loan - LoRA")
print(f"✅ Distilled Fine-tuning completed — ROC-AUC: {auc_distilled:.4f}")

---

## xAI

In [ ]:
import torch
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Load the best (distilled) model
model_path = "loan_model_distilled"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tok = AutoTokenizer.from_pretrained(model_path)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Since your X_test is already text:
print(f"Loaded {len(X_test)} test samples. Example:\n", X_test[0])

# Prediction function: returns probability for each class
def predict_proba_text(texts):
    enc = tok(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt",
        max_length=256
    ).to(device)
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
    return probs  # shape: (N, 2)

class_names = ["Not Approved", "Approved"]

## LIME:

In [ ]:
from lime.lime_text import LimeTextExplainer
import random
import os

random.seed(42)

# Pick a few random samples from X_test
sample_indices = random.sample(range(len(X_test)), k=3)
samples_to_explain = [X_test[i] for i in sample_indices]

print("Explaining samples:", sample_indices)

explainer = LimeTextExplainer(
    class_names=class_names,
    split_expression=r"\|",  # split by the '|' separator
    bow=False,               # use position-aware features
    random_state=42
)

lime_html_paths = []
for idx, text in zip(sample_indices, samples_to_explain):
    print(f"\n🔹 Explaining sample index {idx}...")
    exp = explainer.explain_instance(
        text_instance=text,
        classifier_fn=predict_proba_text,
        labels=[0, 1],
        num_features=10,
        num_samples=1000
    )

    try:
        from IPython.display import display
        display(exp.show_in_notebook(text=True))
    except Exception:
        # Fallback: textual summary if display() is unavailable
        print("\nInline visualization not supported here.")
        for feature, weight in exp.as_list(label=1):  # label 1 = Approved
            print(f"  {feature:50s}  ->  {weight:+.3f}")

    html_path = f"lime_explanation_{idx}.html"
    exp.save_to_file(html_path)
    lime_html_paths.append(os.path.abspath(html_path))

print("✅ LIME explanations saved:", lime_html_paths)


## SHAPE:

In [ ]:
import random, os, webbrowser, numpy as np
import torch
import shap

random.seed(42)
sample_indices = random.sample(range(len(X_test)), k=3)
samples_to_explain = [X_test[i] for i in sample_indices]
print("Explaining SHAP samples:", sample_indices)

def predict_proba_text_safe(texts):
    # Ensure we always have a list of plain strings
    if isinstance(texts, np.ndarray):
        texts = texts.tolist()
    if isinstance(texts, str):
        texts = [texts]
    elif isinstance(texts, (list, tuple)):
        texts = [str(t) for t in texts]
    else:
        texts = [str(texts)]

    enc = tok(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt",
        max_length=256
    ).to(device)

    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
    return probs


try:
    masker = shap.maskers.Text()
    explainer = shap.Explainer(predict_proba_text_safe, masker)
    shap_values = explainer(samples_to_explain)
except Exception as e:
    print("⚠️ SHAP Text masker unavailable:", e)
    print("Falling back to KernelExplainer (slower)...")
    # KernelExplainer expects background as small list of strings
    background = random.sample(X_test, k=min(3, len(X_test)))
    explainer = shap.KernelExplainer(predict_proba_text_safe, background)
    shap_values = explainer.shap_values(samples_to_explain, nsamples=200)

try:
    shap.plots.text(shap_values[..., 1])  # class 1 = Approved
except Exception as e:
    print(f"Inline SHAP visualization skipped: {e}")
    try:
        html_path = "shap_explanations.html"
        shap.save_html(html_path, shap.plots.text(shap_values[..., 1], display=False))
        print(f"✅ SHAP explanations saved to {os.path.abspath(html_path)}")
        webbrowser.open(os.path.abspath(html_path))
    except Exception as e2:
        print(f"Failed to save SHAP HTML: {e2}")
        print("\nFallback textual summary:\n")
        try:
            vals = shap_values.values[..., 1] if hasattr(shap_values, "values") else shap_values[1]
            for i, (txt, v) in enumerate(zip(samples_to_explain, vals)):
                print(f"\n--- Sample {sample_indices[i]} ---")
                print(txt)
                print("Top contributing tokens (approx):")
                parts = np.array(txt.split("|"))
                top_idx = np.argsort(-np.abs(v))[:5]
                for token, weight in zip(parts[top_idx], v[top_idx]):
                    print(f"  {token.strip():50s}  ->  {weight:+.4f}")
        except Exception as e3:
            print("Textual fallback also failed:", e3)


## Final Sample Trustworthy Prediction

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lime.lime_text import LimeTextExplainer
import shap

# Pick a sample index to inspect 
sample_idx = 15
text_sample = X_test[sample_idx]
print(f"🔹 User #{sample_idx}:\n{text_sample}\n")

probs = predict_proba_text([text_sample])[0]
pred_class = np.argmax(probs)
print(f"✅ Model Prediction: {class_names[pred_class]} (prob = {probs[pred_class]:.3f})")

lime_explainer = LimeTextExplainer(class_names=class_names, split_expression=r"\|", bow=False)

lime_exp = lime_explainer.explain_instance(
    text_instance=text_sample,
    classifier_fn=predict_proba_text,
    labels=[0, 1],
    num_features=10,
    num_samples=1000
)

print("\n🔍 Top 10 LIME feature weights:")
for feat, weight in lime_exp.as_list(label=1):
    print(f"  {feat.strip():50s} -> {weight:+.4f}")

try:
    from IPython.display import display
    display(lime_exp.show_in_notebook(text=True))
except Exception:
    html_path = f"lime_explanation_{sample_idx}.html"
    lime_exp.save_to_file(html_path)
    print(f"💾 LIME HTML visualization saved: {html_path}")


def predict_proba_text_safe(texts):
    # Ensure SHAP always receives list[str]
    if isinstance(texts, np.ndarray):
        texts = texts.tolist()
    if isinstance(texts, str):
        texts = [texts]
    elif isinstance(texts, (list, tuple)):
        texts = [str(t) for t in texts]
    else:
        texts = [str(texts)]

    enc = tok(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt",
        max_length=256
    ).to(device)

    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
    return probs

try:
    masker = shap.maskers.Text()
    explainer = shap.Explainer(predict_proba_text_safe, masker)
    shap_values = explainer([text_sample])
    shap.plots.text(shap_values[..., 1])
except Exception as e:
    print(f"⚠️ SHAP Text masker unavailable ({e}). Using KernelExplainer fallback.")
    background = [X_test[0]]
    explainer = shap.KernelExplainer(predict_proba_text_safe, background)
    shap_vals = explainer.shap_values([text_sample], nsamples=200)
    
    # Simplify visualization as a bar chart
    vals = shap_vals[1][0]
    tokens = np.array(text_sample.split("|"))
    df = pd.DataFrame({"token": [t.strip() for t in tokens], "impact": vals[: len(tokens)]})
    df = df.reindex(df.impact.abs().sort_values(ascending=False).index)[:10]

    plt.figure(figsize=(8, 4))
    colors = ["steelblue" if v > 0 else "salmon" for v in df.impact]
    plt.barh(df.token, df.impact, color=colors)
    plt.title("SHAP feature contributions (class: Approved)")
    plt.xlabel("SHAP value")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()